### Wide vs Narrow Transformations

[video](https://youtu.be/761SQ9Hxbic?list=PLeo1K3hjS3uu7dS3Cx0X796sWUjjBFCuV&t=4569)


In [0]:
df = spark.table("workspace.default.movies")
df.show(5)

### Narrow Transformation
each output partition depends on a single input partition - no shuffle (e.g., select, filter, withColumn)

In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.movies")
df_narrow = df.select("title","studio","imdb_rating").filter(F.col("release_year") >= 2010)


df_narrow.explain("extended")  # no Exchange

In [0]:
df_narrow.explain("formatted")

In [0]:
display(df_narrow)

### Wide Transformation
Output partitions depends on multiple input partitions - requires a shuffle/Exchange between stages (e.g., groupBy, non-broadcast join, orderBy)

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 4)

wide1 = (
    df.where(F.col("release_year") >= 2010)
      .repartition(4)
      .groupBy("studio")
      .agg(F.round(F.avg(F.col("revenue").cast("double")), 2).alias("avg_revenue"))
)
wide1.explain("formatted")

In [0]:
display(wide1)